# Necessary conditions and goodness of fit

The answers to Section 7 of the sample questions, `documentation/tex/questions`, run.
The questions concern the compensator of Section 1.3.1 and its closed form for the exponential
kernel (Proposition 1.3.12, Remark 1.3.13), Meyer's theorem (Theorem 1.3.5) and the test of
goodness of fit of Section 1.3.5, with what it does not see (Remark 1.3.46).
The two types are read as the market buys (type 0) and the market sells (type 1).

Each question's code is copied here verbatim, wrapped in a function where it prints.
`outcome` returns what a call prints, or the name of the exception it raises.
`verify` evaluates every choice of the question and asserts that the key holds and that no
other choice does.
A Monte Carlo choice is decided against a standard error: a value within four standard errors
of the one a choice names is close to it, and the notebook prints how many standard errors
separate it from the values the other choices name.

In [ ]:
import contextlib
import io
import timeit
from itertools import islice

import numpy as np
import pandas as pd
from scipy import stats
from scipy.integrate import quad_vec
from scipy.linalg import expm

from unito26.lob import config
from unito26.lob.hawkes import (
    ExponentialHawkes, HawkesParams, OgataThinningHawkes, compensators_at_events,
    intensities_at_events,
)
from unito26.lob.messages import (
    BUY, SELL, GridDepth, limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook


def outcome(function, *arguments):
    """What a call prints, or the name of the exception it raises."""
    printed = io.StringIO()
    try:
        with contextlib.redirect_stdout(printed):
            function(*arguments)
    except Exception as error:
        return type(error).__name__
    return printed.getvalue().strip()


def verify(label, key, holds):
    """Assert that, of the choices of a question, the one lettered `key` holds and no other does."""
    holding = [letter for letter, held in zip("ABCDE", holds) if held]
    assert holding == [key], f"{label}: {holding or 'no choice'} hold, the key is {key}"
    print(f"{label}: ({key}) holds, and no other choice does")


def batch_standard_error(sample, batches=100):
    """The standard error of the mean of a serially dependent sample, by batch means."""
    means = sample[: sample.size // batches * batches].reshape(batches, -1).mean(axis=1)
    return means.std(ddof=1) / np.sqrt(batches)

## `question.palmMeanOfTheTotalIntensity`: the intensity read at the events

The column sums of $A$ are 3, so $\lambda_{\mathfrak g} = 0.5 + 3\,\mathbf 1^\top Z$, and by
(1.3.29) $\mathrm{Var}(\mathbf 1^\top Z) = \bar\lambda/(2\beta(1-\rho)) = 1$.
The mean of $\lambda_{\mathfrak g}$ over the events is $E[\lambda_{\mathfrak g}^2]/\bar\lambda
= (9 + 4)/2 = 6.5$.  The standard error of the event rate is that of $N(T)/T$, from the
stationary count covariance; that of the two means over the events is by batch means.

In [ ]:
def run():
    params = config.self_exciting_pair_params()
    path = np.array(list(ExponentialHawkes(params, rng=3).events(4e4)))
    times, types = path[:, 0], path[:, 1].astype(int)
    before = intensities_at_events(params, times, types).sum(axis=1)
    after = before + params.excitation.sum(axis=0)[types]
    print(round(len(times) / 4e4, 1), round(before.mean(), 1),
          round(after.mean(), 1))


params = config.self_exciting_pair_params()
print("column sums of A:", params.excitation.sum(axis=0), " rho:", params.branching_ratio,
      " lambda*:", params.stationary_intensity())
print("Var(1'Z) by the scalar formula:",
      params.stationary_intensity().sum() / (2 * params.decay * (1 - params.branching_ratio)),
      "  by the Lyapunov equation:", params.stationary_covariance().sum())
path = np.array(list(ExponentialHawkes(params, rng=3).events(4e4)))
times, types = path[:, 0], path[:, 1].astype(int)
before = intensities_at_events(params, times, types).sum(axis=1)
after = before + params.excitation.sum(axis=0)[types]
rate, rate_error = len(times) / 4e4, np.sqrt(params.count_covariance_rate().sum() / 4e4)
mean_before, mean_after = before.mean(), after.mean()
error = batch_standard_error(before)
assert np.allclose(after - before, 3.0)
print(outcome(run))
print(f"rate {rate:.4f} (SE {rate_error:.4f}); mean before {mean_before:.4f}, "
      f"after {mean_after:.4f} (SE {error:.4f})")
for value in (2.0, 2.5, 4.5, 6.5):
    print(f"  mean before the jump against {value}: {(mean_before - value) / error:+.1f} SE")


def near(estimate, value, standard_error):
    return abs(estimate - value) < 4 * standard_error


def reads(rate_value, before_value, after_value):
    return (near(rate, rate_value, rate_error) and near(mean_before, before_value, error)
            and near(mean_after, after_value, error))


verify("question.palmMeanOfTheTotalIntensity", "B", [
    reads(2.0, 2.0, 5.0),
    reads(2.0, 6.5, 9.5),
    reads(2.0, 2.5, 5.5),
    reads(2.0, 4.5, 7.5),
])

## `question.exponentialReferenceOfAPoissonFlow`: the reference law of the gaps

The first positional argument of `stats.expon` is `loc`, and its scale is the mean, $1/3$
for a rate of 3.  The compensator of a Poisson flow of rate 3 is $3t$.

In [ ]:
def run():
    gaps = np.random.default_rng(7).exponential(1 / 3, size=3000)
    references = [
        (gaps, stats.expon(scale=1 / 3).cdf),
        (gaps, stats.expon(3).cdf),
        (3 * gaps, stats.expon().cdf),
        (gaps, stats.expon(scale=3).cdf),
        (gaps, stats.expon().cdf),
    ]
    print([k for k, (x, cdf) in enumerate(references)
           if stats.kstest(x, cdf).pvalue > 0.01])


gaps = np.random.default_rng(7).exponential(1 / 3, size=3000)
for k, (x, cdf) in enumerate([
    (gaps, stats.expon(scale=1 / 3).cdf), (gaps, stats.expon(3).cdf),
    (3 * gaps, stats.expon().cdf), (gaps, stats.expon(scale=3).cdf), (gaps, stats.expon().cdf),
]):
    test = stats.kstest(x, cdf)
    print(f"reference {k}: D = {test.statistic:.4f}, p = {test.pvalue:.3g}")
printed = outcome(run)
print(printed)
verify("question.exponentialReferenceOfAPoissonFlow", "D", [
    printed == "[0, 1, 2]",
    printed == "[2, 3]",
    printed == "[0, 2, 4]",
    printed == "[0, 2]",
    printed == "[2]",
])

## `question.compensatorInMilliseconds`: the unit of time

Remark 1.3.11: the path on a clock $c$ times faster is the process with $\mu/c$, $A/c$,
$\beta/c$.  Read on milliseconds with the rates per second, the state has decayed by
$e^{-4 \cdot 1000\,\Delta t}$ between events, and the mean residual of the buys is
$(1000\,\mu_0 T + \Gamma_{00}N_0 + \Gamma_{01}N_1)/N_0 \approx 250.75$.  A choice names a value
for each of the four lines; a line is near its value within 15%, and the values the choices
name differ by a factor of four or more.

In [ ]:
def run():
    params = config.self_exciting_pair_params()
    path = np.array(list(ExponentialHawkes(params, rng=5).events(4000.0)))
    times, types = path[:, 0], path[:, 1].astype(int)
    mu, A, beta = params.baseline, params.excitation, params.decay
    readings = [
        (times, params),
        (1000 * times, params),
        (1000 * times, HawkesParams(mu / 1000, A / 1000, beta / 1000)),
        (1000 * times, HawkesParams(mu / 1000, A / 1000, beta)),
    ]
    for clock, p in readings:
        compensators = compensators_at_events(p, clock, types)
        tau = np.diff(compensators[types == 0, 0], prepend=0.0)
        print(round(tau.mean(), 2))


params = config.self_exciting_pair_params()
path = np.array(list(ExponentialHawkes(params, rng=5).events(4000.0)))
times, types = path[:, 0], path[:, 1].astype(int)
buys, sells = np.count_nonzero(types == 0), np.count_nonzero(types == 1)
last_buy = times[types == 0][-1]
print(f"{buys} buys, {sells} sells; predicted second line "
      f"{(1000 * 0.25 * last_buy + 0.7 * buys + 0.05 * sells) / buys:.2f}")
printed = outcome(run)
lines = [float(line) for line in printed.split("\n")]
print(lines)


def about(values):
    return all(abs(line - value) < 0.15 * value for line, value in zip(lines, values))


verify("question.compensatorInMilliseconds", "A", [
    about([1, 250, 1, 0.25]),
    about([1, 1, 1, 1]),
    about([1, 250, 1, 1]),
    about([1, 1000, 1, 0.25]),
])

## `question.compensatorReadAfterTheJump`: the record moved past the jump

Alternative B records after both updates, Alternative C after the update of the count alone.

In [ ]:
def compensators_b(params, times, types):
    mu, A, beta = params.baseline, params.excitation, params.decay
    decayed, counts = np.zeros(2), np.zeros(2)
    previous, out = 0.0, np.empty((times.size, 2))
    for k, (time, event_type) in enumerate(zip(times, types)):
        decayed *= np.exp(-beta * (time - previous))
        decayed[event_type] += 1.0
        counts[event_type] += 1.0
        out[k] = mu * time + A @ (counts - decayed) / beta
        previous = time
    return out

In [ ]:
def compensators_c(params, times, types):
    mu, A, beta = params.baseline, params.excitation, params.decay
    decayed, counts = np.zeros(2), np.zeros(2)
    previous, out = 0.0, np.empty((times.size, 2))
    for k, (time, event_type) in enumerate(zip(times, types)):
        decayed *= np.exp(-beta * (time - previous))
        counts[event_type] += 1.0
        out[k] = mu * time + A @ (counts - decayed) / beta
        decayed[event_type] += 1.0
        previous = time
    return out

In [ ]:
def run():
    params = config.self_exciting_pair_params()
    path = np.array(list(ExponentialHawkes(params, rng=1).events(2000.0)))
    times, types = path[:, 0], path[:, 1].astype(int)

    def buy_residuals(compensators):
        return np.diff(compensators[types == 0, 0], prepend=0.0)

    reference = buy_residuals(compensators_at_events(params, times, types))
    for variant in (compensators_b, compensators_c):
        shift = buy_residuals(variant(params, times, types)) - reference
        moved = np.flatnonzero(np.abs(shift) > 1e-9)
        print(moved[:3].tolist(), shift[moved[:3]].round(2).tolist())


print("Gamma:", config.self_exciting_pair_params().branching_matrix.tolist())
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.compensatorReadAfterTheJump", "C", [
    first == second == "[] []",
    first == second == "[0] [0.7]",
    first == "[] []" and second == "[0] [0.7]",
    first == "[] []" and second == "[0, 1, 2] [0.7, 0.7, 0.7]",
    first == "[] []" and second == "[0, 1, 2] [0.7, 1.4, 2.1]",
])

## `question.compensatorByRecursionSumOrQuadrature`: three ways to the compensator

Alternative A is `compensators_at_events`.  The three are first checked to agree, then timed
on the first $n$ events of one path of the self-exciting pair, for $n$ from 2000 to 16000,
in five interleaved rounds, the least time of each kept.  The choices are read off the growth of each time
over the last doubling, and off the ratios to A at the longest path.

In [ ]:
def compensators_by_sum(params, times, types):
    mu, A, beta = params.baseline, params.excitation, params.decay
    out = np.empty((times.size, 2))
    for k, time in enumerate(times):
        mass = (1 - np.exp(-beta * (time - times[:k]))) / beta
        out[k] = mu * time + A[:, types[:k]] @ mass
    return out

In [ ]:
def compensators_by_quadrature(params, times, types):
    mu, A, beta = params.baseline, params.excitation, params.decay
    state, total, previous = np.zeros(2), np.zeros(2), 0.0
    out = np.empty((times.size, 2))
    for k, (time, event_type) in enumerate(zip(times, types)):
        def intensity(u):
            return mu + A @ state * np.exp(-beta * (u - previous))
        total = total + quad_vec(intensity, previous, time)[0]
        out[k] = total
        state = state * np.exp(-beta * (time - previous))
        state[event_type] += 1.0
        previous = time
    return out

In [ ]:
params = config.self_exciting_pair_params()
path = list(islice(ExponentialHawkes(params, rng=4).events(np.inf), 16_000))
path_times = np.array([time for time, _ in path])
path_types = np.array([event_type for _, event_type in path])
reference = compensators_at_events(params, path_times[:2000], path_types[:2000])
for variant in (compensators_by_sum, compensators_by_quadrature):
    np.testing.assert_allclose(
        variant(params, path_times[:2000], path_types[:2000]), reference, rtol=1e-9
    )


def seconds(function, n):
    return timeit.timeit(lambda: function(params, path_times[:n], path_types[:n]), number=1)


ladder = (2_000, 4_000, 8_000, 16_000)
implementations = {
    "A": compensators_at_events, "B": compensators_by_sum, "C": compensators_by_quadrature,
}
best = {(name, n): np.inf for name in implementations for n in ladder}
for _ in range(5):
    for n in ladder:
        for name, function in implementations.items():
            best[name, n] = min(best[name, n], seconds(function, n))
timings = pd.DataFrame(
    [(n, best["A", n], best["B", n], best["C", n]) for n in ladder],
    columns=["events", "A (s)", "B (s)", "C (s)"],
).set_index("events")
for name in "ABC":
    timings[f"{name} growth"] = timings[f"{name} (s)"] / timings[f"{name} (s)"].shift()
timings["B / A"] = timings["B (s)"] / timings["A (s)"]
timings["C / A"] = timings["C (s)"] / timings["A (s)"]
print(timings.round(4).to_string())

growth = {name: timings[f"{name} growth"].iloc[-1] for name in "ABC"}
longest = timings.iloc[-1]
linear = lambda name: growth[name] < 2.5
quadratic = lambda name: growth[name] > 3
verify("question.compensatorByRecursionSumOrQuadrature", "D", [
    longest["C (s)"] < longest["A (s)"] and longest["C (s)"] < longest["B (s)"],
    linear("A") and linear("B") and linear("C"),
    linear("A") and linear("B") and quadratic("C"),
    longest["B / A"] > 5 and longest["C / A"] > 5
    and linear("A") and quadratic("B") and linear("C"),
    longest["B / A"] < 1 / 5 and quadratic("A"),
])

## `question.groundResidualsCannotSeeTheTypes`: one test for the ground process, one for the buys

Both kernels have column sums 3, so both give $\lambda_{\mathfrak g} = 0.5 + 3\,\mathbf 1^\top Z$
on every path: the ground compensator is the same function of the path.

In [ ]:
def run():
    self_exciting = config.self_exciting_pair_params()
    cross_exciting = config.cross_exciting_pair_params()
    events = ExponentialHawkes(self_exciting, rng=1).events(2000.0)
    path = np.array(list(events))
    times, types = path[:, 0], path[:, 1].astype(int)
    for params in (self_exciting, cross_exciting):
        compensators = compensators_at_events(params, times, types)
        ground = np.diff(compensators.sum(axis=1), prepend=0.0)
        buys = np.diff(compensators[types == 0, 0], prepend=0.0)
        print(stats.kstest(ground, "expon").pvalue > 0.01,
              stats.kstest(buys, "expon").pvalue > 0.01)


self_exciting = config.self_exciting_pair_params()
cross_exciting = config.cross_exciting_pair_params()
print("A, self:", self_exciting.excitation.tolist(), " cross:", cross_exciting.excitation.tolist())
path = np.array(list(ExponentialHawkes(self_exciting, rng=1).events(2000.0)))
times, types = path[:, 0], path[:, 1].astype(int)
grounds = []
for name, p in (("self", self_exciting), ("cross", cross_exciting)):
    compensators = compensators_at_events(p, times, types)
    ground = np.diff(compensators.sum(axis=1), prepend=0.0)
    buys = np.diff(compensators[types == 0, 0], prepend=0.0)
    grounds.append(ground)
    print(f"{name}: ground p = {stats.kstest(ground, 'expon').pvalue:.3g}, "
          f"buys mean {buys.mean():.3f}, p = {stats.kstest(buys, 'expon').pvalue:.3g}")
print("ground residuals equal under both kernels:", np.allclose(grounds[0], grounds[1]))
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.groundResidualsCannotSeeTheTypes", "B", [
    first == second == "True True",
    first == "True True" and second == "True False",
    first == "True True" and second == "False False",
    first == "True True" and second == "False True",
])

## `question.antitheticWaitsPassTheMargin`: antithetic uniforms

The correlation of $-\log U$ and $-\log(1-U)$ is $1 - \pi^2/6$, checked first.  Half the
adjacent pairs of Alternative B are antithetic and half independent.

In [ ]:
def waits_a(rng, n):
    return -np.log(rng.random(n)) / 2

def waits_b(rng, n):
    u = rng.random(n // 2)
    return -np.log(np.column_stack([u, 1 - u]).ravel()) / 2

In [ ]:
def run():
    for waits in (waits_a, waits_b):
        tau = 2 * waits(np.random.default_rng(11), 4000)
        r = np.corrcoef(tau[:-1], tau[1:])[0, 1]
        print(stats.kstest(tau, "expon").pvalue > 0.01, f"{r:+.1f}")


u = np.random.default_rng(0).random(10**6)
print(f"1 - pi^2/6 = {1 - np.pi**2 / 6:.4f}; "
      f"sample correlation {np.corrcoef(-np.log(u), -np.log(1 - u))[0, 1]:.4f}")
for waits in (waits_a, waits_b):
    tau = 2 * waits(np.random.default_rng(11), 4000)
    print(f"{waits.__name__}: KS p = {stats.kstest(tau, 'expon').pvalue:.3f}, "
          f"lag-one autocorrelation {np.corrcoef(tau[:-1], tau[1:])[0, 1]:+.4f}")
first, second = outcome(run).split("\n")
print(first, "|", second)
verify("question.antitheticWaitsPassTheMargin", "E", [
    first == second == "True +0.0",
    first == "True +0.0" and second == "False -0.6",
    first == "True +0.0" and second == "True -0.6",
    first == "True +0.0" and second == "False -0.3",
    first == "True +0.0" and second == "True -0.3",
])

## `question.tradesReportedOnBothSides`: common jumps across the two types

In [ ]:
def run():
    rng = np.random.default_rng(14)
    model = [rng.exponential(1.0, 3000).cumsum() for _ in range(2)]
    trades = rng.exponential(1.0, 3000).cumsum()
    for buys, sells in (model, (trades, trades), (trades, trades + 0.001)):
        tau_buy = np.diff(buys, prepend=0.0)
        tau_sell = np.diff(sells, prepend=0.0)
        print(stats.kstest(tau_buy, "expon").pvalue > 0.01,
              stats.kstest(tau_sell, "expon").pvalue > 0.01,
              round(abs(np.corrcoef(tau_buy, tau_sell)[0, 1]), 1))


lines = outcome(run).split("\n")
print(lines)
verify("question.tradesReportedOnBothSides", "C", [
    lines == ["True True 0.0"] * 3,
    lines == ["True True 0.0", "True True 1.0", "True True 0.0"],
    lines == ["True True 0.0", "True True 1.0", "True True 1.0"],
    lines == ["True True 0.0", "False False 1.0", "True True 1.0"],
])

## `question.fillsSharingATimestamp`: one market buy, two executions

The fills give an atom of mass one half at zero: $D = 1/2$.  The orders, rescaled at the rate
of the fills, are $2\,\mathrm{Exp}(1)$: $D = \max_x (e^{-x/2} - e^{-x}) = 1/4$.  A line is near
a named value within 0.04, about twice the standard deviation of $D$ on 2000 draws, and close
to zero below 0.05.

In [ ]:
def run():
    rng = np.random.default_rng(9)
    orders = rng.exponential(1.0, 2000).cumsum()
    fills = np.repeat(orders, 2)
    horizon = orders[-1]
    readings = [
        (fills, len(fills) / horizon),
        (np.unique(fills), len(fills) / horizon),
        (np.unique(fills), len(np.unique(fills)) / horizon),
    ]
    for times, rate in readings:
        tau = rate * np.diff(times, prepend=0.0)
        print(round(stats.kstest(tau, "expon").statistic, 3))


statistics = [float(line) for line in outcome(run).split("\n")]
print(statistics)


def reads(*values):
    return all(abs(s - v) < (0.05 if v == 0 else 0.04) for s, v in zip(statistics, values))


verify("question.fillsSharingATimestamp", "A", [
    reads(0.5, 0.25, 0.0),
    reads(0.0, 0.0, 0.0),
    reads(0.5, 0.5, 0.5),
    reads(0.5, 0.0, 0.0),
    reads(0.25, 0.25, 0.0),
])

## `question.twoSampleTestOnTheGapsOfOnePath`: two correct simulators, compared on one path each

A rejection rate is about 0.01 within four standard errors of the level,
$\sqrt{0.01 \cdot 0.99/300} = 0.0057$, and well above it beyond ten.

In [ ]:
def run():
    def gaps(simulator):
        times = np.array([time for time, _ in simulator.events(250.0)])
        return np.diff(times)

    def rejection_rate(params):
        rejected = 0
        for seed in range(300):
            exact = gaps(ExponentialHawkes(params, rng=seed))
            thinned = gaps(OgataThinningHawkes(params, rng=seed + 1000))
            rejected += stats.ks_2samp(exact, thinned).pvalue < 0.01
        return rejected / 300

    poisson = HawkesParams([1.0, 1.0], np.zeros((2, 2)), 4.0)
    print(rejection_rate(config.self_exciting_pair_params()),
          rejection_rate(poisson))


printed = outcome(run)
self_exciting_rate, poisson_rate = (float(value) for value in printed.split())
level_error = np.sqrt(0.01 * 0.99 / 300)
print(printed, f"   standard error at the level {level_error:.4f}")
print(f"self-exciting: {(self_exciting_rate - 0.01) / level_error:+.1f} SE from the level; "
      f"Poisson: {(poisson_rate - 0.01) / level_error:+.1f} SE")
at_level = lambda rate: abs(rate - 0.01) < 4 * level_error
above = lambda rate: rate > 0.01 + 10 * level_error
verify("question.twoSampleTestOnTheGapsOfOnePath", "B", [
    at_level(self_exciting_rate) and at_level(poisson_rate),
    above(self_exciting_rate) and at_level(poisson_rate),
    above(self_exciting_rate) and above(poisson_rate),
    at_level(self_exciting_rate) and above(poisson_rate),
])